# 02 - Generate real agent trajectories with AgentDojo + vLLM (Days 11, 13, 14, 18)
Settings: **GPU T4 x2**, **Internet on**, secret `HF_TOKEN`. One model per session.

| model | vLLM tool parser |
|---|---|
| Qwen/Qwen2.5-7B-Instruct-AWQ (model A) | hermes |
| hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4 (model B) | llama3_json |

Budget: roughly 600 attacked tasks per attack per model; sessions die after 9-12 h, so run one suite per cell
and upload logs after each.

In [ ]:
!pip -q install vllm agentdojo openai
!git clone -q https://github.com/<you>/tracewarden.git && pip -q install -e tracewarden
MODEL = 'Qwen/Qwen2.5-7B-Instruct-AWQ'; PARSER = 'hermes'; TAG = 'qwen'

In [ ]:
import subprocess, time, requests
srv = subprocess.Popen(['vllm', 'serve', MODEL, '--quantization', 'awq', '--dtype', 'half',
                        '--max-model-len', '16384', '--gpu-memory-utilization', '0.90',
                        '--tensor-parallel-size', '2', '--enable-auto-tool-choice', '--tool-call-parser', PARSER,
                        '--port', '8000'], stdout=open('vllm.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        if requests.get('http://localhost:8000/v1/models').ok: break
    except Exception: pass
    time.sleep(10)
print(open('vllm.log').read()[-1500:])

In [ ]:
# ALWAYS check the flags of your installed AgentDojo first: model names, local-model support, module loading
!python -m agentdojo.scripts.benchmark --help

In [ ]:
import os
os.environ['OPENAI_BASE_URL'] = 'http://localhost:8000/v1'; os.environ['OPENAI_API_KEY'] = 'sk-none'
os.environ['TW_LLM_BASE_URL'] = 'http://localhost:8000/v1'; os.environ['TW_LLM_MODEL'] = MODEL
SUITES = ['banking', 'slack', 'travel', 'workspace']
# Recent AgentDojo versions support an OpenAI-compatible local server via `--model local --model-id <hf id>`.
# If yours does not, register the endpoint as a custom model following AgentDojo's docs.
BASE = f'python -m agentdojo.scripts.benchmark --model local --model-id {MODEL}'

In [ ]:
# benign runs (no attack) and single-step attacks
for s in SUITES:
    !{BASE} -s {s} --logdir /kaggle/working/logs_{TAG}_benign
    !{BASE} -s {s} --attack important_instructions --logdir /kaggle/working/logs_{TAG}_ii

In [ ]:
# N7 multi-step decomposed and N8 translated attacks (custom attacks from scripts/agentdojo_attacks.py)
%cd tracewarden
for s in SUITES:
    !{BASE} -s {s} --attack multistep_decomposed --module-to-load scripts.agentdojo_attacks --logdir /kaggle/working/logs_{TAG}_multistep
for lang in ['ur', 'roman_ur']:
    !{BASE} -s banking -s slack --attack translated_{lang} --module-to-load scripts.agentdojo_attacks --logdir /kaggle/working/logs_{TAG}_{lang}
%cd ..

In [ ]:
# convert + auto-label right here, then upload both raw logs and labeled jsonl
!tracewarden data agentdojo --root /kaggle/working/logs_{TAG}_ii --out /kaggle/working/dojo_{TAG}/ii --name agentdojo-{TAG} --split test --identity keep
!tracewarden data agentdojo --root /kaggle/working/logs_{TAG}_benign --out /kaggle/working/dojo_{TAG}/benign --name agentdojo-{TAG} --split test --identity keep
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login
login(UserSecretsClient().get_secret('HF_TOKEN'))
api = HfApi(); repo = '<you>/agentdojo-steps-raw'
api.create_repo(repo, repo_type='dataset', private=True, exist_ok=True)
api.upload_folder(folder_path='/kaggle/working', repo_id=repo, repo_type='dataset',
                  allow_patterns=['logs_*/**', 'dojo_*/**'])